In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/final_dataset.csv")

df.head()

,district,year,avg_aqi,population,density,literacy,healthcare_facilities
0,Central,2020,338.22,582320,27730,85.1,32
1,Central,2021,249.49,582320,27730,85.1,32
2,Central,2022,243.10,582320,27730,85.1,32
3,Central,2023,196.15,582320,27730,85.1,32
4,East,2017,375.93,1709346,27132,89.3,19


In [2]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

df["aqi_score"] = scaler.fit_transform(df[["avg_aqi"]]) * 100

df["density_score"] = scaler.fit_transform(df[["density"]]) * 100

df["healthcare_score"] = scaler.fit_transform(
    df[["healthcare_facilities"]]
) * 100

df["literacy_score"] = scaler.fit_transform(
    df[["literacy"]]
) * 100

In [3]:
df[
    [
        "aqi_score",
        "density_score",
        "healthcare_score",
        "literacy_score"
    ]
].describe()

,aqi_score,density_score,healthcare_score,literacy_score
count,29.000000,29.000000,29.000000,29.000000
mean,31.816297,52.997357,61.685824,53.553835
std,22.839638,42.203975,37.952003,38.980795
min,0.000000,0.000000,0.000000,0.000000
25%,12.492666,12.600969,61.111111,14.285714
50%,34.969685,63.350386,72.222222,53.061224
75%,40.245453,97.316460,88.888889,79.591837
max,100.000000,100.000000,100.000000,100.000000


In [4]:
df["healthcare_risk"] = 100 - df["healthcare_score"]

df["literacy_risk"] = 100 - df["literacy_score"]

In [5]:
df["urban_pulse_risk_score"] = (
    0.40 * df["aqi_score"]
    + 0.30 * df["density_score"]
    + 0.20 * df["healthcare_risk"]
    + 0.10 * df["literacy_risk"]
)

In [6]:
def risk_category(score):

    if score < 33:
        return "Low"

    elif score < 66:
        return "Medium"

    else:
        return "High"


df["risk_category"] = df[
    "urban_pulse_risk_score"
].apply(risk_category)

In [7]:
df[
    [
        "district",
        "year",
        "avg_aqi",
        "urban_pulse_risk_score",
        "risk_category"
    ]
].sort_values(
    "urban_pulse_risk_score",
    ascending=False
).head(15)

,district,year,avg_aqi,urban_pulse_risk_score,risk_category
4,East,2017,375.93,89.194938,High
0,Central,2020,338.22,76.751666,High
5,East,2018,294.13,73.196503,High
9,East,2022,276.97,69.840352,High
10,East,2023,263.15,67.137438,High
8,East,2021,251.85,64.927385,Medium
6,East,2019,251.35,64.829595,Medium
1,Central,2021,249.49,59.397862,Medium
2,Central,2022,243.10,58.148107,Medium
7,East,2020,212.79,57.288034,Medium


In [8]:
df.to_csv(
    "../data/processed/final_scored_dataset.csv",
    index=False
)

print("Saved successfully")

Saved successfully


In [9]:
df["urban_pulse_risk_score"].describe()

count    29.000000
mean     40.933177
std      23.363731
min       9.818594
25%      18.777357
50%      41.098215
75%      59.397862
max      89.194938
Name: urban_pulse_risk_score, dtype: float64

In [10]:
df["risk_category"].value_counts()

risk_category
Medium    12
Low       12
High       5
Name: count, dtype: int64